# Rendering the guidance sweep at scale

The earlier settings sweep measured classifier-free guidance on ten images per value. This notebook
renders the first 200 of the shared 500 COCO captions at six guidance values with the tuned Stable
Diffusion 1.5 setting (Euler-ancestral, 30 steps, leading spacing), enough for FID and Inception
Score to show how image quality trades against prompt adherence. Nothing is tuned or trained here;
the images are scored in a separate evaluation notebook.

## 1. Setup

Captions are read straight from the attached COCO Dataset, and images, plots and tables go to
`/kaggle/working/outputs/`. The model weights download from the Hugging Face Hub, so the session
needs internet access.

In [ ]:
import gc, os, textwrap, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import diffusers
import transformers
from diffusers import (EulerAncestralDiscreteScheduler, PNDMScheduler, StableDiffusionPipeline)
from transformers import CLIPTextModel, CLIPTokenizer

warnings.filterwarnings("ignore", category=FutureWarning)
transformers.logging.set_verbosity_error()
diffusers.logging.set_verbosity_error()


# The Dataset is read in place from /kaggle/input; nothing is copied or written under it.
def resolve_paths():
    if Path("/kaggle/input").is_dir() and Path("/kaggle/working").is_dir():
        found = sorted(Path("/kaggle/input").glob("**/processed/pairs.parquet"))
        if not found:
            raise RuntimeError(
                "Attach the Dataset holding processed/pairs.parquet. Under /kaggle/input: "
                + (", ".join(p.name for p in Path("/kaggle/input").iterdir()) or "nothing"))
        return found[0].parent, Path("/kaggle/working/outputs")
    for root in (Path("."), Path("..")):
        if (root / "data" / "processed" / "pairs.parquet").is_file():
            return root / "data" / "processed", root / "outputs" / "milestone_3"
    raise RuntimeError("data/processed/pairs.parquet not found -- run from the repo root")


PROCESSED, OUT = resolve_paths()
IMAGES, PLOTS, TABLES = OUT / "images", OUT / "plots", OUT / "tables"
for d in (IMAGES, PLOTS, TABLES):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"

print(f"torch {torch.__version__} | diffusers {diffusers.__version__} | transformers {transformers.__version__}")
print(f"device  : {DEVICE}  {GPU_NAME}")
if DEVICE.type == "cuda":
    print(f"memory  : {torch.cuda.get_device_properties(0).total_memory / 1024 ** 3:.1f} GB")
print(f"data    : {PROCESSED}")
print(f"outputs : {OUT}")

### Settings

`M3_SMOKE=1` in the environment renders 4 prompts at guidance 1 and 7.5 only. Guidance 1 switches
classifier-free guidance off, so the pipeline skips the unconditional pass and that value costs
half as much. Above 12 the guidance rescale of 0.7 from the earlier sweep is applied, so the
g = 15 point moves two settings at once.

In [ ]:
SEED = 42
SMOKE = os.environ.get("M3_SMOKE", "0") == "1"
N_PROMPTS = 4 if SMOKE else 200
BATCH_SIZE = 4
GUIDANCE_GRID = (1, 7.5) if SMOKE else (1, 3, 5, 7.5, 11, 15)
GUIDANCE_RESCALE = 0.7
RESCALE_ABOVE = 12
CLIP_MAX_TOKENS = 77

SD_REPOS = ["stable-diffusion-v1-5/stable-diffusion-v1-5",
            "sd-legacy/stable-diffusion-v1-5",
            "CompVis/stable-diffusion-v1-4"]
CLIP_SD_MODEL = "openai/clip-vit-large-patch14"
SD_SIZE = 512
SAMPLER, STEPS, TIMESTEP_SPACING = "euler_a", 30, "leading"

CONFIG_ORDER = [f"sweep_g{g:g}" for g in GUIDANCE_GRID]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"smoke {SMOKE} | {N_PROMPTS} prompts | guidance {GUIDANCE_GRID} | batch {BATCH_SIZE} | {DTYPE}")
print("folders:", CONFIG_ORDER)

## 2. The prompt set

Sampled exactly as in the models notebook, which renders the first 500; this one takes the first
200, so `idx` means the same caption in both.

In [ ]:
pairs = pd.read_parquet(PROCESSED / "pairs.parquet")
sample = pairs[pairs.cap_rank == 0].sample(n=500, random_state=42).reset_index(drop=True)
sample["idx"] = range(len(sample))

prompts = sample.head(N_PROMPTS).copy()
prompts["image_512_path"] = prompts["image_512_path"].map(lambda p: Path(p).name)
prompts[["idx", "image_id", "caption", "image_512_path"]].to_csv(TABLES / "m3_prompts_guidance.csv", index=False)

IDX = prompts["idx"].tolist()
CAPTIONS = prompts["caption"].tolist()
assert IDX == list(range(len(IDX)))

print(f"{len(pairs):,} pairs -> {len(sample)} sampled -> {len(prompts)} rendered here")
print(f"wrote {TABLES / 'm3_prompts_guidance.csv'}\n")
for i, caption in zip(IDX[:5], CAPTIONS[:5]):
    print(f"  {i:03d}  {caption}")

## 3. Helpers

Each prompt has its own CPU generator seeded `SEED + idx`, used for its starting latent and for the
noise Euler-ancestral draws at every step. Every guidance value therefore starts prompt *i* from the
same noise, and guidance 7.5 here reproduces the models notebook's `sd_tuned` images.

Rendering is resumable: an image already on disk is skipped, and the log is rewritten after every
guidance value. After a resumed run the timing columns cover only the latest session.

In [ ]:
LOG_PATH = TABLES / "m3_sweep_log.csv"
LOG_COLUMNS = ["config", "model", "encoder", "sampler", "steps", "guidance", "resolution",
               "n_images", "seconds_total", "sec_per_image", "peak_vram_gb"]
log_rows = ({row["config"]: row for row in pd.read_csv(LOG_PATH).to_dict("records")}
            if LOG_PATH.is_file() else {})


def batches(items, size):
    return [items[i:i + size] for i in range(0, len(items), size)]


def prompt_generator(idx):
    return torch.Generator(device="cpu").manual_seed(SEED + idx)


# Starting noise for prompt idx, plus its generator with the stream advanced past that
# noise, so the in-loop noise of an ancestral sampler does not repeat the starting noise.
def start_noise(idx, shape):
    generator = prompt_generator(idx)
    return torch.randn((1, *shape), generator=generator), generator


def image_path(config, idx):
    return IMAGES / config / f"{idx:03d}.png"


def save_image(img, path):
    partial = path.with_name(path.stem + ".partial")
    img.convert("RGB").save(partial, format="PNG")
    partial.replace(path)


def free():
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


def write_log():
    order = [c for c in CONFIG_ORDER if c in log_rows] + [c for c in log_rows if c not in CONFIG_ORDER]
    pd.DataFrame([log_rows[c] for c in order])[LOG_COLUMNS].to_csv(LOG_PATH, index=False)


def render_config(config, meta, render_batch, batch_size=BATCH_SIZE):
    (IMAGES / config).mkdir(parents=True, exist_ok=True)
    todo = [i for i in IDX if not image_path(config, i).exists()]
    print(f"{config}: {len(IDX) - len(todo)} already on disk, {len(todo)} to render")
    if todo:
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        started = time.time()
        for part in tqdm(batches(todo, batch_size), desc=config):
            for i, img in zip(part, render_batch(part)):
                save_image(img, image_path(config, i))
        seconds = time.time() - started
        peak = torch.cuda.max_memory_allocated() / 1024 ** 3 if DEVICE.type == "cuda" else float("nan")
        log_rows[config] = {"config": config, **meta,
                            "seconds_total": round(seconds, 1),
                            "sec_per_image": round(seconds / len(todo), 3),
                            "peak_vram_gb": round(peak, 2)}
    elif config not in log_rows:
        log_rows[config] = {"config": config, **meta, "seconds_total": float("nan"),
                            "sec_per_image": float("nan"), "peak_vram_gb": float("nan")}
    log_rows[config]["n_images"] = sum(image_path(config, i).exists() for i in IDX)
    write_log()
    row = log_rows[config]
    print(f"  {row['n_images']} images, {row['sec_per_image']} s/image, peak {row['peak_vram_gb']} GB")

## 4. Text embeddings

CLIP ViT-L/14's last layer, encoded once in float32 and passed to SD as `prompt_embeds`, with the
empty string as the negative. Captions are encoded one at a time, exactly as in the models notebook,
because batched encoding changes the fifth decimal with batch size and that is enough to change an
Euler-ancestral image.

In [ ]:
# transformers 4 nests the layer norm under .text_model; transformers 5 does not.
def final_layer_norm(text_encoder):
    return getattr(text_encoder, "text_model", text_encoder).final_layer_norm


@torch.no_grad()
def encode_prompts(texts, tokenizer, text_encoder, layer="last", batch_size=1):
    out = []
    for part in batches(list(texts), batch_size):
        tokens = tokenizer(part, padding="max_length", max_length=CLIP_MAX_TOKENS,
                           truncation=True, return_tensors="pt").to(DEVICE)
        hidden = text_encoder(input_ids=tokens["input_ids"], output_hidden_states=True)
        if layer == "last":
            states = hidden.last_hidden_state
        else:
            states = final_layer_norm(text_encoder)(hidden.hidden_states[-2])
        out.append(states.float().cpu())
    return torch.cat(out)


tokenizer = CLIPTokenizer.from_pretrained(CLIP_SD_MODEL)
text_encoder = CLIPTextModel.from_pretrained(CLIP_SD_MODEL).to(DEVICE, torch.float32).eval()


POS = encode_prompts(CAPTIONS, tokenizer, text_encoder)
NEG = encode_prompts([""], tokenizer, text_encoder)
assert POS.shape == (len(CAPTIONS), CLIP_MAX_TOKENS, 768), POS.shape
print(f"prompt {tuple(POS.shape)}  negative {tuple(NEG.shape)}")

del text_encoder, tokenizer
free()

## 5. The guidance sweep

Attention slicing is left off. The earlier notebooks enabled it for the 8 GB laptop GPU, but at
batch 4 PyTorch's built-in attention already peaks under 3 GB, and slicing made each image about
40% slower in a local check. VAE slicing stays on.

In [ ]:
SCHEDULER_CLASSES = {"pndm": PNDMScheduler, "euler_a": EulerAncestralDiscreteScheduler}


def load_sd(repos=SD_REPOS):
    errors = {}
    for repo in repos:
        try:
            pipe = StableDiffusionPipeline.from_pretrained(
                repo, torch_dtype=DTYPE, safety_checker=None, requires_safety_checker=False)
        except Exception as exc:
            errors[repo] = f"{type(exc).__name__}: {exc}"
            continue
        pipe = pipe.to(DEVICE)
        pipe.set_progress_bar_config(disable=True)
        pipe.vae.enable_slicing()
        return pipe, repo, dict(pipe.scheduler.config)
    raise RuntimeError("no Stable Diffusion repo could be loaded: " + repr(errors))


# Swaps always start from the scheduler config SD 1.5 shipped with, never the live one.
def set_scheduler(pipe, name):
    pipe.scheduler = SCHEDULER_CLASSES[name].from_config(BASE_SCHEDULER_CONFIG,
                                                         timestep_spacing=TIMESTEP_SPACING)


def sd_renderer(pos, neg, steps, guidance, rescale=0.0):
    def render(part):
        noise = [start_noise(i, LATENT_SHAPE) for i in part]
        out = pipe(prompt_embeds=pos[part].to(DEVICE, DTYPE),
                   negative_prompt_embeds=neg.expand(len(part), -1, -1).to(DEVICE, DTYPE),
                   latents=torch.cat([n for n, _ in noise]).to(DEVICE, DTYPE),
                   generator=[g for _, g in noise],
                   num_inference_steps=steps, guidance_scale=guidance, guidance_rescale=rescale,
                   height=SD_SIZE, width=SD_SIZE)
        return out.images
    return render


t0 = time.time()
pipe, SD_REPO, BASE_SCHEDULER_CONFIG = load_sd()
LATENT_SHAPE = (pipe.unet.config.in_channels, SD_SIZE // pipe.vae_scale_factor, SD_SIZE // pipe.vae_scale_factor)
print(f"loaded {SD_REPO} in {time.time() - t0:.1f}s | latents {LATENT_SHAPE} | "
      f"beta_schedule {BASE_SCHEDULER_CONFIG['beta_schedule']}")


set_scheduler(pipe, SAMPLER)
print(f"scheduler {type(pipe.scheduler).__name__} | spacing {pipe.scheduler.config.timestep_spacing}")

for g, name in zip(GUIDANCE_GRID, CONFIG_ORDER):
    rescale = GUIDANCE_RESCALE if g > RESCALE_ABOVE else 0.0
    meta = {"model": SD_REPO, "encoder": "CLIP ViT-L/14 last layer", "sampler": SAMPLER,
            "steps": STEPS, "guidance": g, "resolution": SD_SIZE}
    if rescale:
        print(f"{name}: guidance_rescale {rescale}")
    render_config(name, meta, sd_renderer(POS, NEG, STEPS, g, rescale))

del pipe
free()

## 6. Does guidance 7.5 match `sd_tuned`?

`sweep_g7.5` and the models notebook's `sd_tuned` use the same setting, prompts and noise, so their
shared images should agree to within float16 noise. The check looks for `sd_tuned` in this
session's outputs and in any attached Kaggle input, and is skipped when neither has it. On a
different GPU, small differences are expected.

In [ ]:
reference_dirs = [IMAGES / "sd_tuned"]
if Path("/kaggle/input").is_dir():
    reference_dirs += sorted(Path("/kaggle/input").glob("**/images/sd_tuned"))

compared = []
for ref_dir in reference_dirs:
    for i in IDX[:3]:
        ref, ours = ref_dir / f"{i:03d}.png", image_path("sweep_g7.5", i)
        if ref.is_file() and ours.is_file():
            a = np.asarray(Image.open(ref).convert("RGB"), dtype=np.int16)
            b = np.asarray(Image.open(ours).convert("RGB"), dtype=np.int16)
            diff = np.abs(a - b)
            compared.append({"reference": str(ref_dir), "idx": i,
                             "mean_abs_diff": round(float(diff.mean()), 3), "max_abs_diff": int(diff.max())})
    if compared:
        break

if compared:
    check = pd.DataFrame(compared)
    print(check.to_string(index=False))
    close = bool((check["mean_abs_diff"] < 2.0).all())
    print(f"\npixel-close (mean |diff| < 2 of 255): {close}")
else:
    print("no sd_tuned images found next to this run or under /kaggle/input -- check skipped")

## 7. Log, preview and timing

The preview shows the first four prompts across the guidance values, left to right from no guidance
to the strongest.

In [ ]:
def preview_grid(configs, labels, path, idxs):
    fig, axes = plt.subplots(len(idxs), len(configs), figsize=(2.4 * len(configs), 2.5 * len(idxs)),
                             squeeze=False)
    for r, i in enumerate(idxs):
        for c, (config, label) in enumerate(zip(configs, labels)):
            ax = axes[r, c]
            ax.set_xticks([]); ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)
            if image_path(config, i).exists():
                ax.imshow(Image.open(image_path(config, i)))
            if r == 0:
                ax.set_title(label, fontsize=9)
            if c == 0:
                ax.set_ylabel(textwrap.fill(CAPTIONS[i], 26), rotation=0, ha="right",
                              va="center", fontsize=7, labelpad=8)
    fig.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()


def time_chart(log, path, title):
    fig, ax = plt.subplots(figsize=(7.5, 0.55 * len(log) + 1.4))
    ax.barh(log["config"], log["sec_per_image"], color="#4C72B0")
    ax.invert_yaxis()
    for y, (s, total) in enumerate(zip(log["sec_per_image"], log["seconds_total"])):
        if pd.notna(s):
            ax.text(s, y, f"  {s:.2f} s/img, {total / 60:.1f} min", va="center", fontsize=8)
    ax.set_xlim(0, np.nanmax(log["sec_per_image"].to_numpy(dtype=float)) * 1.45)
    ax.set_xlabel("seconds per image (this session)")
    ax.set_title(title, fontsize=10)
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, dpi=120, bbox_inches="tight")
    plt.show()


log = pd.read_csv(LOG_PATH)
print(log.to_string(index=False))
print(f"\ntotal rendering time this session: {log['seconds_total'].sum() / 60:.1f} min")

counts = {c: len(list((IMAGES / c).glob("*.png"))) for c in CONFIG_ORDER}
print("images on disk:", counts)

labels = [f"g = {g:g}" + (" (rescale 0.7)" if g > RESCALE_ABOVE else "") for g in GUIDANCE_GRID]
preview_grid(CONFIG_ORDER, labels, PLOTS / "m3_render_guidance_preview.png", IDX[:4])
time_chart(log, PLOTS / "m3_render_guidance_time.png", f"Rendering cost per guidance value ({GPU_NAME})")
print(PLOTS / "m3_render_guidance_preview.png")
print(PLOTS / "m3_render_guidance_time.png")

## 8. What was rendered

Six folders of 200 PNGs at 512 px under `outputs/images/`: `sweep_g1`, `sweep_g3`, `sweep_g5`,
`sweep_g7.5`, `sweep_g11` and `sweep_g15`, one file per prompt named by its index (`000.png` to
`199.png`). `tables/m3_prompts_guidance.csv` maps each index to its caption and COCO image,
`tables/m3_sweep_log.csv` records the settings and cost of each guidance value, and the preview and
timing plots are in `plots/`.

The evaluation notebook computes FID, Inception Score and CLIP score for each guidance value from
these files. Attach this notebook's output to it as a Kaggle input.